# Notebook 04 — Front YOLOv8n Validation Evaluation

**Objective:** reload and evaluate the accepted Front YOLOv8n baseline on the 254-image validation split. All reported Precision, Recall, and mAP values are **VALIDATION METRICS**, not test accuracy, final accuracy, or independent test performance.

This notebook does not train, export, or modify the model or dataset. It does not perform Notebook 05.

## 1. Experiment metadata, environment, and CONFIG

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, logging, math, os, platform, shutil, subprocess, sys, time, warnings
import numpy as np
import pandas as pd
import torch
import ultralytics
import yaml

EXPERIMENT_ID = 'FRONT_DET_YOLOV8N_EVAL_001'
STARTED_AT = datetime.now(timezone.utc).isoformat()
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
CONDA_ENV = os.environ.get('CONDA_DEFAULT_ENV', '')
GIT_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_ROOT, capture_output=True, text=True, check=True).stdout.strip()
CUDA_AVAILABLE = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else 'NOT_AVAILABLE'
MODEL_PATH = PROJECT_ROOT / 'runs' / 'front' / 'yolov8n_front_v1_baseline' / 'weights' / 'best.pt'
EXPECTED_MODEL_SHA256 = '750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'roboflow' / 'front_detect_v1'
DATA_YAML = DATASET_ROOT / 'data.yaml'
VALID_IMAGES_DIR = DATASET_ROOT / 'valid' / 'images'
VALID_LABELS_DIR = DATASET_ROOT / 'valid' / 'labels'
TRAINING_METRICS_PATH = PROJECT_ROOT / 'results' / 'detection' / 'front_yolov8n_baseline_metrics.csv'
IMGSZ, DEVICE = 640, 0
COUNT_DIAGNOSTIC_CONF = 0.25
REPRO_WARNING_THRESHOLD = 0.01
REPRO_FAIL_THRESHOLD = 0.05
EXPECTED_VALIDATION_IMAGES, EXPECTED_VALIDATION_OBJECTS = 254, 995
EVAL_OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'front' / 'detection' / 'evaluation'
EVAL_RUN_NAME = EXPERIMENT_ID
EVAL_RUN_DIR_EXPECTED = EVAL_OUTPUT_ROOT / EVAL_RUN_NAME
CONFIG = {'experiment_id': EXPERIMENT_ID, 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'expected_model_sha256': EXPECTED_MODEL_SHA256, 'data': str(DATA_YAML.relative_to(PROJECT_ROOT)), 'dataset_version': 1, 'evaluation_split': 'valid', 'imgsz': IMGSZ, 'device': DEVICE, 'count_diagnostic_conf': COUNT_DIAGNOSTIC_CONF, 'repro_warning_threshold': REPRO_WARNING_THRESHOLD, 'repro_fail_threshold': REPRO_FAIL_THRESHOLD}
print(f'experiment_id: {EXPERIMENT_ID}')
print(f'datetime_utc: {STARTED_AT}')
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Python executable: {sys.executable}')
print(f'Conda environment: {CONDA_ENV}')
print(f'Python: {platform.python_version()}')
print(f'Torch: {torch.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'CUDA available: {CUDA_AVAILABLE}; CUDA runtime: {torch.version.cuda}; GPU: {GPU_NAME}')
print(f'Git commit: {GIT_COMMIT}')
print('CONFIG')
print(yaml.safe_dump(CONFIG, sort_keys=False))

experiment_id: FRONT_DET_YOLOV8N_EVAL_001
datetime_utc: 2026-08-17T06:01:49.494965+00:00
PROJECT_ROOT: /home/diy-hus/fish
Python executable: /home/diy-hus/miniconda3/envs/fish/bin/python
Conda environment: fish
Python: 3.11.15
Torch: 2.13.0+cu130
Ultralytics: 8.4.120
CUDA available: True; CUDA runtime: 13.0; GPU: NVIDIA GeForce RTX 3050
Git commit: 7e23b580488fe650fb3b379ca97f686d5b8ce9af
CONFIG
experiment_id: FRONT_DET_YOLOV8N_EVAL_001
model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
expected_model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
data: data/roboflow/front_detect_v1/data.yaml
dataset_version: 1
evaluation_split: valid
imgsz: 640
device: 0
count_diagnostic_conf: 0.25
repro_warning_threshold: 0.01
repro_fail_threshold: 0.05



## 2. Mandatory preflight: model hash and validation dataset

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''): digest.update(chunk)
    return digest.hexdigest()
assert CONDA_ENV == 'fish', f'FAIL preflight: expected Conda env fish, found {CONDA_ENV!r}'
assert CUDA_AVAILABLE, 'FAIL preflight: CUDA is required because DEVICE=0.'
assert MODEL_PATH.is_file(), f'FAIL preflight: missing model {MODEL_PATH}'
MODEL_SHA256 = sha256_file(MODEL_PATH)
assert MODEL_SHA256 == EXPECTED_MODEL_SHA256, f'FAIL preflight: model SHA-256 mismatch: {MODEL_SHA256}'
assert DATA_YAML.is_file(), f'FAIL preflight: missing {DATA_YAML}'
assert VALID_IMAGES_DIR.is_dir() and VALID_LABELS_DIR.is_dir(), 'FAIL preflight: validation images/labels structure is missing.'
assert TRAINING_METRICS_PATH.is_file(), f'FAIL preflight: missing {TRAINING_METRICS_PATH}'
with DATA_YAML.open(encoding='utf-8') as handle: DATA_DEFINITION = yaml.safe_load(handle)
CLASS_NAMES_RAW = DATA_DEFINITION.get('names')
CLASS_NAMES = {index: name for index, name in enumerate(CLASS_NAMES_RAW)} if isinstance(CLASS_NAMES_RAW, list) else {int(key): value for key, value in CLASS_NAMES_RAW.items()}
assert CLASS_NAMES == {0: 'Ca'}, f'FAIL preflight: unexpected classes {CLASS_NAMES}'
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}
VALID_IMAGE_PATHS = sorted(path for path in VALID_IMAGES_DIR.rglob('*') if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
VALID_LABEL_PATHS = sorted(VALID_LABELS_DIR.rglob('*.txt'))
VALIDATION_OBJECTS = sum(sum(bool(line.strip()) for line in path.read_text(encoding='utf-8', errors='replace').splitlines()) for path in VALID_LABEL_PATHS)
assert len(VALID_IMAGE_PATHS) == EXPECTED_VALIDATION_IMAGES, f'FAIL preflight: expected 254 validation images, found {len(VALID_IMAGE_PATHS)}'
assert VALIDATION_OBJECTS == EXPECTED_VALIDATION_OBJECTS, f'FAIL preflight: expected 995 validation objects, found {VALIDATION_OBJECTS}'
if EVAL_RUN_DIR_EXPECTED.exists() and any(EVAL_RUN_DIR_EXPECTED.iterdir()):
    raise RuntimeError(f'FAIL preflight: preserve existing evaluation output and choose a new approved run name: {EVAL_RUN_DIR_EXPECTED}')
MODEL_SIZE_MB = MODEL_PATH.stat().st_size / (1024 ** 2)
print(f'Model: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
print(f'Model SHA-256: {MODEL_SHA256}')
print(f'Model size: {MODEL_SIZE_MB:.3f} MB')
print(f'data.yaml: {DATA_YAML.relative_to(PROJECT_ROOT)}')
print(f'Validation images: {len(VALID_IMAGE_PATHS)}; validation objects: {VALIDATION_OBJECTS}; classes: {CLASS_NAMES}')
print('PREFLIGHT_RESULT: PASS')

Model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
Model SHA-256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
Model size: 5.954 MB
data.yaml: data/roboflow/front_detect_v1/data.yaml
Validation images: 254; validation objects: 995; classes: {0: 'Ca'}
PREFLIGHT_RESULT: PASS


## 3. Reload best.pt and run Ultralytics validation

Ultralytics logs remain visible. Default validation thresholds are not tuned. Plots—including confusion matrix and available PR/F1/P/R curves—are saved under the ignored `outputs/front/detection/evaluation/` directory.

In [3]:
from ultralytics import YOLO
from ultralytics.utils import LOGGER
class WarningCollector(logging.Handler):
    def __init__(self): super().__init__(level=logging.WARNING); self.messages = []
    def emit(self, record): self.messages.append(self.format(record))
warning_collector = WarningCollector()
LOGGER.addHandler(warning_collector)
print(f'Loading accepted model: {MODEL_PATH.relative_to(PROJECT_ROOT)}')
MODEL_OBJECT = YOLO(str(MODEL_PATH), task='detect')
MODEL_INFO_RAW = MODEL_OBJECT.info(verbose=True, imgsz=IMGSZ)
MODEL_PARAMETERS = int(sum(parameter.numel() for parameter in MODEL_OBJECT.model.parameters()))
MODEL_GFLOPS = float(MODEL_INFO_RAW[-1]) if isinstance(MODEL_INFO_RAW, tuple) and MODEL_INFO_RAW and isinstance(MODEL_INFO_RAW[-1], (int, float)) else None
VAL_START = time.perf_counter()
try:
    with warnings.catch_warnings(record=True) as captured_warnings:
        warnings.simplefilter('always')
        VAL_RESULTS = MODEL_OBJECT.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, device=DEVICE, plots=True, project=str(EVAL_OUTPUT_ROOT), name=EVAL_RUN_NAME, exist_ok=False, verbose=True)
        PYTHON_WARNINGS = [str(item.message) for item in captured_warnings]
finally:
    LOGGER.removeHandler(warning_collector)
VALIDATION_RUNTIME_SEC = time.perf_counter() - VAL_START
VALIDATION_WARNINGS = list(dict.fromkeys(warning_collector.messages + PYTHON_WARNINGS))
EVAL_RUN_DIR = Path(VAL_RESULTS.save_dir).resolve()
PRECISION = float(VAL_RESULTS.box.mp)
RECALL = float(VAL_RESULTS.box.mr)
MAP50 = float(VAL_RESULTS.box.map50)
MAP50_95 = float(VAL_RESULTS.box.map)
CLASS_MAP50_95 = np.atleast_1d(VAL_RESULTS.box.maps).astype(float)
CLASS_PRECISION = np.atleast_1d(VAL_RESULTS.box.p).astype(float)
CLASS_RECALL = np.atleast_1d(VAL_RESULTS.box.r).astype(float)
SPEED = {key: float(value) for key, value in VAL_RESULTS.speed.items()}
PREPROCESS_MS = SPEED.get('preprocess', float('nan'))
INFERENCE_MS = SPEED.get('inference', float('nan'))
POSTPROCESS_MS = SPEED.get('postprocess', float('nan'))
INFERENCE_FPS = 1000.0 / INFERENCE_MS if INFERENCE_MS > 0 else float('nan')
print('VALIDATION METRICS')
print(f'Precision: {PRECISION:.6f}; Recall: {RECALL:.6f}; mAP50: {MAP50:.6f}; mAP50-95: {MAP50_95:.6f}')
print(f'Speed ms/image: preprocess={PREPROCESS_MS:.3f}, inference={INFERENCE_MS:.3f}, postprocess={POSTPROCESS_MS:.3f}')
print(f'Model-only inference FPS estimate: {INFERENCE_FPS:.3f} (not end-to-end video FPS)')
print(f'Validation output: {EVAL_RUN_DIR.relative_to(PROJECT_ROOT)}')
print(f'Validation warnings ({len(VALIDATION_WARNINGS)}):')
for message in VALIDATION_WARNINGS: print(f'- {message}')

Loading accepted model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
Model summary: 130 layers, 3,011,043 parameters, 0 gradients, 8.2 GFLOPs
Ultralytics 8.4.120 🚀 Python-3.11.15 torch-2.13.0+cu130 CUDA:0 (NVIDIA GeForce RTX 3050, 8192MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 114.4±28.0 MB/s, size: 55.6 KB)
val: Scanning /home/diy-hus/fish/data/roboflow/front_detect_v1/valid/labels.cache... 254 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 254/254 76.1Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 10, len(boxes) = 995. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 5.5it/s 2.9s0.1s
                   all  

## 4. Reproducibility check against Notebook 03 training validation

In [4]:
TRAINING_METRICS = pd.read_csv(TRAINING_METRICS_PATH)
assert len(TRAINING_METRICS) == 1, 'FAIL: expected one Notebook 03 baseline metrics row.'
metric_pairs = [('precision', PRECISION), ('recall', RECALL), ('mAP50', MAP50), ('mAP50_95', MAP50_95)]
REPRODUCIBILITY_ROWS = []
for metric, current_value in metric_pairs:
    training_value = float(TRAINING_METRICS.iloc[0][metric])
    REPRODUCIBILITY_ROWS.append({'metric': metric, 'training_validation': training_value, 'reloaded_best_model_validation': current_value, 'absolute_difference': abs(current_value - training_value)})
REPRODUCIBILITY_DF = pd.DataFrame(REPRODUCIBILITY_ROWS)
MAX_METRIC_DIFFERENCE = float(REPRODUCIBILITY_DF['absolute_difference'].max())
display(REPRODUCIBILITY_DF)
print(f'Maximum absolute difference vs training validation: {MAX_METRIC_DIFFERENCE:.8f}')
if MAX_METRIC_DIFFERENCE > REPRO_FAIL_THRESHOLD: print('REPRODUCIBILITY_RESULT: FAIL')
elif MAX_METRIC_DIFFERENCE > REPRO_WARNING_THRESHOLD: print('REPRODUCIBILITY_RESULT: PASS_WITH_WARNING')
else: print('REPRODUCIBILITY_RESULT: PASS')

,metric,training_validation,reloaded_best_model_validation,absolute_difference
0,precision,0.96682,0.967795,0.000975
1,recall,0.96630,0.966466,0.000166
2,mAP50,0.98976,0.989832,0.000072
3,mAP50_95,0.55233,0.552514,0.000184


Maximum absolute difference vs training validation: 0.00097495
REPRODUCIBILITY_RESULT: PASS


## 5. Fixed-confidence prediction diagnostics

Predictions at `COUNT_DIAGNOSTIC_CONF = 0.25` support confidence and per-image count diagnostics only. They do not replace Precision/Recall/mAP and are not a calibration study.

In [5]:
DIAGNOSTIC_DIR = EVAL_OUTPUT_ROOT / 'diagnostics'
FAILURE_CASES_DIR = EVAL_OUTPUT_ROOT / 'failure_cases'
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)
FAILURE_CASES_DIR.mkdir(parents=True, exist_ok=True)
GT_COUNTS = {path.stem: sum(bool(line.strip()) for line in path.read_text(encoding='utf-8', errors='replace').splitlines()) for path in VALID_LABEL_PATHS}
confidence_values, count_rows = [], []
PREDICTION_START = time.perf_counter()
prediction_stream = MODEL_OBJECT.predict(source=[str(path) for path in VALID_IMAGE_PATHS], imgsz=IMGSZ, conf=COUNT_DIAGNOSTIC_CONF, device=DEVICE, stream=True, save=False, verbose=False)
for index, result in enumerate(prediction_stream, 1):
    image_path = Path(result.path)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(float) if result.boxes is not None else np.array([], dtype=float)
    confidence_values.extend(confidences.tolist())
    n_ground_truth, n_predictions = GT_COUNTS.get(image_path.stem, 0), len(confidences)
    count_rows.append({'image_path': str(image_path.relative_to(DATASET_ROOT)), 'n_ground_truth': n_ground_truth, 'n_predictions': n_predictions, 'count_error': n_predictions - n_ground_truth, 'absolute_count_error': abs(n_predictions - n_ground_truth), 'mean_prediction_confidence': float(confidences.mean()) if len(confidences) else float('nan'), 'min_prediction_confidence': float(confidences.min()) if len(confidences) else float('nan')})
    if index % 25 == 0 or index == len(VALID_IMAGE_PATHS): print(f'Prediction diagnostics: {index}/{len(VALID_IMAGE_PATHS)}; elapsed={time.perf_counter() - PREDICTION_START:.1f}s')
COUNT_PER_IMAGE_DF = pd.DataFrame(count_rows)
CONFIDENCE_ARRAY = np.asarray(confidence_values, dtype=float)
assert len(COUNT_PER_IMAGE_DF) == EXPECTED_VALIDATION_IMAGES, 'FAIL: prediction diagnostics did not cover all validation images.'
assert len(CONFIDENCE_ARRAY) > 0, 'FAIL: no predictions available for confidence diagnostics.'
CONFIDENCE_STATS = {'number_of_predictions': int(len(CONFIDENCE_ARRAY)), 'mean': float(CONFIDENCE_ARRAY.mean()), 'median': float(np.median(CONFIDENCE_ARRAY)), 'std': float(CONFIDENCE_ARRAY.std()), 'min': float(CONFIDENCE_ARRAY.min()), 'max': float(CONFIDENCE_ARRAY.max()), 'Q25': float(np.quantile(CONFIDENCE_ARRAY, 0.25)), 'Q75': float(np.quantile(CONFIDENCE_ARRAY, 0.75))}
COUNT_ERRORS = COUNT_PER_IMAGE_DF['count_error'].to_numpy(dtype=float)
COUNT_MAE = float(np.mean(np.abs(COUNT_ERRORS)))
COUNT_RMSE = float(np.sqrt(np.mean(COUNT_ERRORS ** 2)))
COUNT_BIAS = float(np.mean(COUNT_ERRORS))
EXACT_COUNT_RATE = float(np.mean(COUNT_ERRORS == 0))
UNDERCOUNT_RATE = float(np.mean(COUNT_ERRORS < 0))
OVERCOUNT_RATE = float(np.mean(COUNT_ERRORS > 0))
print(f'Confidence diagnostics: {CONFIDENCE_STATS}')
print(f'COUNT DIAGNOSTICS at conf={COUNT_DIAGNOSTIC_CONF}: MAE={COUNT_MAE:.6f}, RMSE={COUNT_RMSE:.6f}, bias={COUNT_BIAS:.6f}, exact={EXACT_COUNT_RATE:.6f}, under={UNDERCOUNT_RATE:.6f}, over={OVERCOUNT_RATE:.6f}')
COUNT_PER_IMAGE_PATH = DIAGNOSTIC_DIR / 'count_diagnostics_per_image.csv'
COUNT_PER_IMAGE_DF.to_csv(COUNT_PER_IMAGE_PATH, index=False)
print(f'Created local diagnostic {COUNT_PER_IMAGE_PATH.relative_to(PROJECT_ROOT)} ({COUNT_PER_IMAGE_PATH.stat().st_size} bytes)')

Prediction diagnostics: 25/254; elapsed=6.0s
Prediction diagnostics: 50/254; elapsed=6.0s
Prediction diagnostics: 75/254; elapsed=6.0s
Prediction diagnostics: 100/254; elapsed=6.0s
Prediction diagnostics: 125/254; elapsed=6.0s
Prediction diagnostics: 150/254; elapsed=6.0s
Prediction diagnostics: 175/254; elapsed=6.0s
Prediction diagnostics: 200/254; elapsed=6.0s
Prediction diagnostics: 225/254; elapsed=6.0s
Prediction diagnostics: 250/254; elapsed=6.0s
Prediction diagnostics: 254/254; elapsed=6.0s
Confidence diagnostics: {'number_of_predictions': 1070, 'mean': 0.7354282786913007, 'median': 0.76812744140625, 'std': 0.10108479081410542, 'min': 0.25608721375465393, 'max': 0.8549965620040894, 'Q25': 0.7233634740114212, 'Q75': 0.7879766970872879}
COUNT DIAGNOSTICS at conf=0.25: MAE=0.326772, RMSE=0.739759, bias=0.295276, exact=0.759843, under=0.015748, over=0.224409
Created local diagnostic outputs/front/detection/evaluation/diagnostics/count_diagnostics_per_image.csv (30317 bytes)


## 6. Confidence histogram and small failure-review set

In [6]:
import matplotlib.pyplot as plt
CONFIDENCE_HISTOGRAM = DIAGNOSTIC_DIR / 'confidence_histogram.png'
plt.figure(figsize=(8, 4.5))
plt.hist(CONFIDENCE_ARRAY, bins=30, color='steelblue', edgecolor='white')
plt.xlabel('Prediction confidence'); plt.ylabel('Predictions'); plt.title(f'Validation confidence diagnostic (conf ≥ {COUNT_DIAGNOSTIC_CONF})')
plt.tight_layout(); plt.savefig(CONFIDENCE_HISTOGRAM, dpi=140); plt.show()
review_candidates = COUNT_PER_IMAGE_DF.sort_values(['absolute_count_error', 'mean_prediction_confidence'], ascending=[False, True]).head(12).copy()
review_candidates['review_reason'] = np.where(review_candidates['count_error'] < 0, 'count_undercount_possible_false_negative', np.where(review_candidates['count_error'] > 0, 'count_overcount_possible_false_positive', 'low_confidence_review'))
FAILURE_REVIEW_CSV = DIAGNOSTIC_DIR / 'failure_review.csv'
review_candidates.to_csv(FAILURE_REVIEW_CSV, index=False)
review_sources = [str(DATASET_ROOT / value) for value in review_candidates['image_path']]
for result in MODEL_OBJECT.predict(source=review_sources, imgsz=IMGSZ, conf=COUNT_DIAGNOSTIC_CONF, device=DEVICE, stream=True, save=False, verbose=False):
    destination = FAILURE_CASES_DIR / Path(result.path).name
    result.save(filename=str(destination))
print(f'Created confidence histogram: {CONFIDENCE_HISTOGRAM.relative_to(PROJECT_ROOT)}')
print(f'Created failure review: {FAILURE_REVIEW_CSV.relative_to(PROJECT_ROOT)}; images={len(review_candidates)}')
print('Failure labels are review categories based on count/confidence diagnostics, not confirmed IoU-matched FP/FN classifications.')

<Figure size 800x450 with 1 Axes>

Created confidence histogram: outputs/front/detection/evaluation/diagnostics/confidence_histogram.png
Created failure review: outputs/front/detection/evaluation/diagnostics/failure_review.csv; images=12
Failure labels are review categories based on count/confidence diagnostics, not confirmed IoU-matched FP/FN classifications.


## 7. Save compact evaluation evidence

In [7]:
LOG_DIR = PROJECT_ROOT / 'logs' / 'detection' / EXPERIMENT_ID
RESULTS_DIR = PROJECT_ROOT / 'results' / 'detection'
LOG_DIR.mkdir(parents=True, exist_ok=True); RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = LOG_DIR / 'config.yaml'
ENVIRONMENT_PATH = LOG_DIR / 'environment.txt'
SUMMARY_PATH = LOG_DIR / 'summary.json'
VALIDATION_METRICS_PATH = RESULTS_DIR / 'front_yolov8n_validation_metrics.csv'
REPRODUCIBILITY_PATH = RESULTS_DIR / 'front_yolov8n_reproducibility_check.csv'
COUNT_DIAGNOSTICS_PATH = RESULTS_DIR / 'front_yolov8n_count_diagnostics.csv'
CONFIG_EVIDENCE = dict(CONFIG, git_commit=GIT_COMMIT, class_names=CLASS_NAMES)
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG_EVIDENCE, sort_keys=False), encoding='utf-8')
ENVIRONMENT_LINES = [f'experiment_id={EXPERIMENT_ID}', f'datetime_utc={STARTED_AT}', f'git_commit={GIT_COMMIT}', f'python_executable={sys.executable}', f'python={platform.python_version()}', f'conda_env={CONDA_ENV}', f'torch={torch.__version__}', f'cuda_runtime={torch.version.cuda}', f'cuda_available={CUDA_AVAILABLE}', f'gpu={GPU_NAME}', f'ultralytics={ultralytics.__version__}']
ENVIRONMENT_PATH.write_text('\n'.join(ENVIRONMENT_LINES) + '\n', encoding='utf-8')
validation_rows = [{'scope': 'overall', 'class_id': None, 'class_name': 'all', 'precision': PRECISION, 'recall': RECALL, 'mAP50': MAP50, 'mAP50_95': MAP50_95}]
for class_id, class_name in CLASS_NAMES.items():
    validation_rows.append({'scope': 'per_class', 'class_id': class_id, 'class_name': class_name, 'precision': float(CLASS_PRECISION[class_id]), 'recall': float(CLASS_RECALL[class_id]), 'mAP50': MAP50 if len(CLASS_NAMES) == 1 else float('nan'), 'mAP50_95': float(CLASS_MAP50_95[class_id])})
pd.DataFrame(validation_rows).to_csv(VALIDATION_METRICS_PATH, index=False)
REPRODUCIBILITY_DF.to_csv(REPRODUCIBILITY_PATH, index=False)
COUNT_DIAGNOSTICS_DF = pd.DataFrame([{**{'count_diagnostic_conf': COUNT_DIAGNOSTIC_CONF, 'validation_images': len(COUNT_PER_IMAGE_DF), 'validation_objects': VALIDATION_OBJECTS, 'count_MAE': COUNT_MAE, 'count_RMSE': COUNT_RMSE, 'count_bias': COUNT_BIAS, 'exact_count_rate': EXACT_COUNT_RATE, 'undercount_rate': UNDERCOUNT_RATE, 'overcount_rate': OVERCOUNT_RATE}, **CONFIDENCE_STATS}])
COUNT_DIAGNOSTICS_DF.to_csv(COUNT_DIAGNOSTICS_PATH, index=False)
EVIDENCE_WARNINGS = list(VALIDATION_WARNINGS)
EVIDENCE_WARNINGS.append('Optional test split is absent; all reported model metrics are VALIDATION METRICS.')
if MAX_METRIC_DIFFERENCE > REPRO_WARNING_THRESHOLD: EVIDENCE_WARNINGS.append(f'Maximum metric difference vs training validation is {MAX_METRIC_DIFFERENCE:.8f}.')
if MAX_METRIC_DIFFERENCE > REPRO_FAIL_THRESHOLD: CHECKPOINT_RESULT = 'FAIL'
elif EVIDENCE_WARNINGS: CHECKPOINT_RESULT = 'PASS_WITH_WARNING'
else: CHECKPOINT_RESULT = 'PASS'
OUTPUT_FILES = [CONFIG_PATH, ENVIRONMENT_PATH, SUMMARY_PATH, VALIDATION_METRICS_PATH, REPRODUCIBILITY_PATH, COUNT_DIAGNOSTICS_PATH]
SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'model_sha256': MODEL_SHA256, 'dataset': str(DATASET_ROOT.relative_to(PROJECT_ROOT)), 'dataset_version': 1, 'evaluation_split': 'valid', 'validation_images': len(VALID_IMAGE_PATHS), 'validation_objects': VALIDATION_OBJECTS, 'precision': PRECISION, 'recall': RECALL, 'mAP50': MAP50, 'mAP50_95': MAP50_95, 'difference_vs_training': REPRODUCIBILITY_DF.to_dict(orient='records'), 'count_MAE': COUNT_MAE, 'count_RMSE': COUNT_RMSE, 'count_bias': COUNT_BIAS, 'exact_count_rate': EXACT_COUNT_RATE, 'undercount_rate': UNDERCOUNT_RATE, 'overcount_rate': OVERCOUNT_RATE, 'model_parameters': MODEL_PARAMETERS, 'model_GFLOPs': MODEL_GFLOPS, 'model_size_MB': MODEL_SIZE_MB, 'imgsz': IMGSZ, 'preprocess_ms': PREPROCESS_MS, 'inference_ms': INFERENCE_MS, 'postprocess_ms': POSTPROCESS_MS, 'inference_fps': INFERENCE_FPS, 'validation_runtime_sec': round(VALIDATION_RUNTIME_SEC, 3), 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': EVIDENCE_WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'local_plot_directory': str(EVAL_OUTPUT_ROOT.relative_to(PROJECT_ROOT)), 'git_commit': GIT_COMMIT, 'next_step': 'USER reviews and accepts Notebook 04 results before any Notebook 05 work.'}
SUMMARY_PATH.write_text(json.dumps(SUMMARY, indent=2, ensure_ascii=False), encoding='utf-8')
for path in OUTPUT_FILES:
    print(f'Created {path.relative_to(PROJECT_ROOT)} ({path.stat().st_size} bytes)')

Created logs/detection/FRONT_DET_YOLOV8N_EVAL_001/config.yaml (459 bytes)
Created logs/detection/FRONT_DET_YOLOV8N_EVAL_001/environment.txt (338 bytes)
Created logs/detection/FRONT_DET_YOLOV8N_EVAL_001/summary.json (2873 bytes)
Created results/detection/front_yolov8n_validation_metrics.csv (240 bytes)
Created results/detection/front_yolov8n_reproducibility_check.csv (307 bytes)
Created results/detection/front_yolov8n_count_diagnostics.csv (462 bytes)


## 8. Final Summary

In [8]:
FINAL_SUMMARY = {'experiment_id': EXPERIMENT_ID, 'model': str(MODEL_PATH.relative_to(PROJECT_ROOT)), 'model_sha256': MODEL_SHA256, 'dataset': str(DATASET_ROOT.relative_to(PROJECT_ROOT)), 'dataset_version': 1, 'evaluation_split': 'valid', 'validation_images': len(VALID_IMAGE_PATHS), 'validation_objects': VALIDATION_OBJECTS, 'precision': PRECISION, 'recall': RECALL, 'mAP50': MAP50, 'mAP50_95': MAP50_95, 'difference_vs_training': REPRODUCIBILITY_DF.to_dict(orient='records'), 'count_MAE': COUNT_MAE, 'count_RMSE': COUNT_RMSE, 'count_bias': COUNT_BIAS, 'exact_count_rate': EXACT_COUNT_RATE, 'model_parameters': MODEL_PARAMETERS, 'model_GFLOPs': MODEL_GFLOPS, 'model_size_MB': MODEL_SIZE_MB, 'preprocess_ms': PREPROCESS_MS, 'inference_ms': INFERENCE_MS, 'postprocess_ms': POSTPROCESS_MS, 'checkpoint_result': CHECKPOINT_RESULT, 'warnings': EVIDENCE_WARNINGS, 'output_files': [str(path.relative_to(PROJECT_ROOT)) for path in OUTPUT_FILES], 'next_step': SUMMARY['next_step']}
print('FINAL SUMMARY')
for key, value in FINAL_SUMMARY.items(): print(f'{key}: {value}')

FINAL SUMMARY
experiment_id: FRONT_DET_YOLOV8N_EVAL_001
model: runs/front/yolov8n_front_v1_baseline/weights/best.pt
model_sha256: 750b0f8a1621f7214c8122467e5360ada69673ae4a8dc5bf3fd7b1e280287738
dataset: data/roboflow/front_detect_v1
dataset_version: 1
evaluation_split: valid
validation_images: 254
validation_objects: 995
precision: 0.9677949492608868
recall: 0.9664655247601828
mAP50: 0.9898324652250613
mAP50_95: 0.5525137653149601
difference_vs_training: [{'metric': 'precision', 'training_validation': 0.96682, 'reloaded_best_model_validation': 0.9677949492608868, 'absolute_difference': 0.0009749492608868149}, {'metric': 'recall', 'training_validation': 0.9663, 'reloaded_best_model_validation': 0.9664655247601828, 'absolute_difference': 0.00016552476018272255}, {'metric': 'mAP50', 'training_validation': 0.98976, 'reloaded_best_model_validation': 0.9898324652250613, 'absolute_difference': 7.246522506132447e-05}, {'metric': 'mAP50_95', 'training_validation': 0.55233, 'reloaded_best_model